# 02 — Files and I/O

## Learning Objectives

By the end of this notebook you will be able to:

- Open, read, and write files using `open()` with different modes
- Use the `with` statement to manage file handles safely
- Read files line by line vs. all at once
- Use `pathlib.Path.read_text()` and `.write_text()` as simpler alternatives
- Avoid common pitfalls: relative paths, encoding issues, forgetting to close files

## Setup

In [ ]:
import os
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains, check_length, check_keys

# Create output directory so exercises have somewhere to write
os.makedirs("output", exist_ok=True)
print("output/ directory ready")

## Opening files with `open()`

The built-in `open()` function opens a file and returns a **file object**.
The second argument is the **mode**:

| Mode | Meaning |
|---|---|
| `"r"` | Read (default). File must exist. |
| `"w"` | Write. Creates file if missing; **overwrites** if it exists. |
| `"a"` | Append. Creates file if missing; **adds to end** if it exists. |
| `"x"` | Exclusive create. Fails if file already exists. |

Always include `encoding="utf-8"` to avoid surprises on Windows.

## The `with` statement — automatic cleanup

Always use `with open(...) as f:` instead of calling `f = open(...)` manually.
The `with` block **automatically closes the file** when the block exits — even if an
exception is raised. Forgetting to close files causes resource leaks.

This is Python's equivalent of Node's `fs.promises` pattern with try/finally.

In [ ]:
# Write a small file as a demonstration
with open("output/demo.txt", "w", encoding="utf-8") as f:
    f.write("model-a-v1\n")
    f.write("model-b-v1\n")
    f.write("model-a-v2\n")

print("File written.")

# Read it back — f.read() returns the entire file as one string
with open("output/demo.txt", "r", encoding="utf-8") as f:
    contents = f.read()

print("Contents:")
print(contents)

## Reading files — three approaches

1. **`f.read()`** — returns the whole file as a single string
2. **`f.readlines()`** — returns a list of strings, one per line (each includes `\n`)
3. **Iterating** — `for line in f:` reads one line at a time (memory-efficient for big files)

In [ ]:
# Approach 1: read the whole file at once
with open("output/demo.txt", "r", encoding="utf-8") as f:
    whole = f.read()
print("f.read() result:", repr(whole))  # repr shows \n explicitly

# Approach 2: read into a list of lines
with open("output/demo.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
print("\nf.readlines():", lines)

# Strip trailing newlines for cleaner values
clean_lines = [line.rstrip("\n") for line in lines]
print("Cleaned:", clean_lines)

# Approach 3: iterate line by line
print("\nLine by line:")
with open("output/demo.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(" -", line.strip())

## Writing files — `write()` and `writelines()`

In [ ]:
models = ["model-a-v1", "model-b-v1", "model-a-v2"]

# writelines() takes a list of strings — you supply the \n yourself
with open("output/models.txt", "w", encoding="utf-8") as f:
    f.writelines(m + "\n" for m in models)

# Verify
with open("output/models.txt", "r", encoding="utf-8") as f:
    print(f.read())

## `pathlib` shortcuts — `.read_text()` and `.write_text()`

For simple cases, `pathlib.Path` has convenient one-liners that handle open/close for you.
These are great when you just need to read or write a whole file's content.

In [ ]:
from pathlib import Path

p = Path("output/pathlib_demo.txt")

# Write the whole file in one call
p.write_text("hello from pathlib\n", encoding="utf-8")

# Read the whole file in one call
text = p.read_text(encoding="utf-8")
print(repr(text))  # 'hello from pathlib\n'

## Common gotchas

**Relative vs absolute paths:** When a notebook runs, its working directory is wherever Jupyter
started — not necessarily the notebook's folder. Use `os.getcwd()` to check. If in doubt,
use `pathlib` relative to `Path(__file__)` in scripts, or construct absolute paths.

**Encoding:** Always specify `encoding="utf-8"`. Omitting it relies on the system default,
which can cause `UnicodeDecodeError` on Windows or when files contain non-ASCII characters
(common in model outputs and prompt data).

**Forgetting `\n`:** `f.write("hello")` writes `hello` without a newline. Use `f.write("hello\n")`
or `print("hello", file=f)` if you want each line on its own line.

## Your Turn — Exercises

### Exercise 1: Write a list of model names to a file, then read it back

Write the three model names below to `"output/test_models.txt"`, one per line.
Then read the file back and store the list of names (stripped of whitespace) in `read_back`.

In [ ]:
model_names = ["gpt-4", "claude-3", "gemini-pro"]

# YOUR CODE HERE — write to "output/test_models.txt"
# YOUR CODE HERE — read back into read_back (a list of strings)
read_back = []

In [ ]:
check_equal(read_back, ["gpt-4", "claude-3", "gemini-pro"], "read_back matches written names")
check_type(read_back, list, "read_back is a list")

### Exercise 2: Count the lines in the file

Read `"output/test_models.txt"` and store the number of non-empty lines in `line_count`.

In [ ]:
# YOUR CODE HERE
line_count = None

In [ ]:
check_equal(line_count, 3, "file has 3 lines")
check_type(line_count, int, "line_count is an int")

### Exercise 3: Append a new model name to the file

Open `"output/test_models.txt"` in append mode and add `"llama-3"` as a new line.
Then read the file back and store all lines (stripped) in `updated_lines`.

In [ ]:
# YOUR CODE HERE — append "llama-3" to the file


# YOUR CODE HERE — read back all lines into updated_lines
updated_lines = []

In [ ]:
check_length(updated_lines, 4, "file now has 4 lines")
check_contains(updated_lines, "llama-3", "llama-3 is in the file")

## Why This Matters for AI Research Engineering

Research results live in files. In a typical AI safety research workflow you will:

- Read files of model outputs (JSON, CSV, plain text) that were generated by evaluation pipelines
- Write analysis results to files so they can be shared, versioned, and reproduced
- Load prompt datasets line-by-line (sometimes millions of lines — memory matters)
- Append results incrementally so a long-running evaluation can resume after failure

Clean file I/O code — using `with` blocks, specifying encodings, and handling paths carefully —
is the foundation of reproducible research.

## Summary

| Concept | Code | Notes |
|---|---|---|
| Open for reading | `open("f.txt", "r")` | Default mode |
| Open for writing | `open("f.txt", "w")` | Overwrites existing |
| Open for appending | `open("f.txt", "a")` | Adds to end |
| Auto-close | `with open(...) as f:` | Always prefer this |
| Read all | `f.read()` | Returns one big string |
| Read lines | `f.readlines()` | Returns list, lines include `\n` |
| Iterate lines | `for line in f:` | Memory-efficient |
| Write string | `f.write(s)` | No automatic `\n` |
| Pathlib shortcut | `Path(p).read_text()` / `.write_text()` | Clean for simple cases |

**Next up:** `03_json_and_csv.ipynb` — loading structured data.